In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle
import time
import os

# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path="outputs/model.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

scale      = input_details[0]['quantization'][0]
zero_point = input_details[0]['quantization'][1]

# Load the 7 EFS features and test data
with open("../data-pipeline/data/processed/minimal_feature_set.pkl", "rb") as f:
    features = pickle.load(f)

X_test = pd.read_csv("../data-pipeline/data/processed/X_test.csv")[features]
y_test = pd.read_csv("../data-pipeline/data/processed/y_test.csv")["class"]

model_size_bytes = os.path.getsize("outputs/model.tflite")

print(f"Model loaded: {model_size_bytes/1024:.2f} KB")
print(f"Test samples: {len(X_test):,}")
print(f"Features:     {features}")


Model loaded: 2.80 KB
Test samples: 354,705
Features:     ['Header_Length', 'Number', 'ack_flag_number', 'TCP', 'ack_count', 'Tot size', 'AVG']


In [3]:
N = 1000
infer_times = []

for i in range(N):
    sample = X_test.iloc[i % len(X_test)].values.reshape(1, -1).astype(np.float32)
    sample_int8 = (sample / scale + zero_point).astype(np.int8)

    start = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
    end = time.perf_counter()

    infer_times.append((end - start) * 1000)  # convert to ms

infer_times = np.array(infer_times)

print(f"TFLite Inference Time ({N} samples):")
print(f"  Mean:   {infer_times.mean():.4f} ms")
print(f"  Min:    {infer_times.min():.4f} ms")
print(f"  Max:    {infer_times.max():.4f} ms")
print(f"  Std:    {infer_times.std():.4f} ms")

TFLite Inference Time (1000 samples):
  Mean:   0.0099 ms
  Min:    0.0030 ms
  Max:    3.7646 ms
  Std:    0.1327 ms


In [4]:
# Reconstruct the rules using the same threshold logic from Phase D
from sklearn.ensemble import RandomForestClassifier

# Load training data to recompute thresholds
X_train = pd.read_csv("../data-pipeline/data/processed/X_train.csv")[features]
y_train = pd.read_csv("../data-pipeline/data/processed/y_train.csv")["class"]

# Compute per-class means (same logic as 05_xai_rules.ipynb)
classes = ['Benign', 'DDoS', 'Reconnaissance']
means = {cls: X_train[y_train == cls].mean() for cls in classes}

rules = []
for feat in features:
    b = means['Benign'][feat]
    d = means['DDoS'][feat]
    r = means['Reconnaissance'][feat]
    rules.append({'feature': feat, 'op': '>' if d > b else '<',
                  'threshold': round((b + d) / 2, 4), 'class': 'DDoS',
                  'explanation': f'DDoS detected: {feat} is abnormal.'})
    rules.append({'feature': feat, 'op': '>' if r > b else '<',
                  'threshold': round((b + r) / 2, 4), 'class': 'Reconnaissance',
                  'explanation': f'Probe detected: {feat} is abnormal.'})
rules.append({'feature': None, 'op': None, 'threshold': None,
              'class': 'Benign', 'explanation': 'No threat detected.'})

def apply_rules(row, rule_list):
    for rule in rule_list:
        if rule['feature'] is None:
            return rule['class'], rule['explanation']
        val = row[rule['feature']]
        if rule['op'] == '>' and val > rule['threshold']:
            return rule['class'], rule['explanation']
        if rule['op'] == '<' and val < rule['threshold']:
            return rule['class'], rule['explanation']
    return 'Benign', 'No threat detected.'

# Time the XAI lookup over 1000 samples
xai_times = []
for i in range(N):
    row = X_test.iloc[i % len(X_test)]
    start = time.perf_counter()
    _ = apply_rules(row, rules)
    end = time.perf_counter()
    xai_times.append((end - start) * 1000)

xai_times = np.array(xai_times)
print(f"XAI Rule Lookup Time ({N} samples):")
print(f"  Mean:   {xai_times.mean():.6f} ms")
print(f"  Min:    {xai_times.min():.6f} ms")
print(f"  Max:    {xai_times.max():.6f} ms")

XAI Rule Lookup Time (1000 samples):
  Mean:   0.009177 ms
  Min:    0.003900 ms
  Max:    0.047200 ms


In [5]:
xai_size = os.path.getsize("outputs/xai_rules.h")
modelh_size = os.path.getsize("outputs/model.h")
tensor_arena = model_size_bytes * 2
stack_bytes  = 30 * 1024
total = model_size_bytes + xai_size + tensor_arena + stack_bytes

print("MODEL SIZE & MEMORY METRICS")
print("=" * 45)
print(f"  model.tflite:        {model_size_bytes/1024:.2f} KB")
print(f"  model.h (C array):   {modelh_size/1024:.2f} KB")
print(f"  xai_rules.h:         {xai_size/1024:.2f} KB")
print(f"  Tensor arena est.:   {tensor_arena/1024:.2f} KB")
print(f"  Framework stack:     {stack_bytes/1024:.2f} KB")
print(f"  TOTAL FOOTPRINT:     {total/1024:.2f} KB")
print()
print(f"{'Device':<30} {'Budget':>8} {'Used':>8} {'Result':>8}")
print("-" * 58)
for name, budget_kb in [("ESP32", 520), ("Raspberry Pi Pico", 264), ("Arduino Nano 33 BLE Sense", 256)]:
    result = "PASS" if total/1024 < budget_kb else "FAIL"
    print(f"{name:<30} {budget_kb:>7}KB {total/1024:>7.2f}KB {result:>8}")

MODEL SIZE & MEMORY METRICS
  model.tflite:        2.80 KB
  model.h (C array):   17.19 KB
  xai_rules.h:         1.38 KB
  Tensor arena est.:   5.61 KB
  Framework stack:     30.00 KB
  TOTAL FOOTPRINT:     39.79 KB

Device                           Budget     Used   Result
----------------------------------------------------------
ESP32                              520KB   39.79KB     PASS
Raspberry Pi Pico                  264KB   39.79KB     PASS
Arduino Nano 33 BLE Sense          256KB   39.79KB     PASS


In [6]:
print("PERFORMANCE BENCHMARKING SUMMARY — CB011911")
print("=" * 55)
print(f"{'Metric':<35} {'Value':<15} {'Target':<10} {'Result'}")
print("-" * 55)
rows = [
    ("TFLite inference time (mean)",
     f"{infer_times.mean():.4f} ms", "< 10 ms",
     "PASS" if infer_times.mean() < 10 else "CHECK"),
    ("TFLite inference time (max)",
     f"{infer_times.max():.4f} ms", "< 50 ms",
     "PASS" if infer_times.max() < 50 else "CHECK"),
    ("XAI rule overhead (mean)",
     f"{xai_times.mean():.6f} ms", "Negligible", "PASS"),
    ("model.tflite size",
     f"{model_size_bytes/1024:.2f} KB", "< 50 KB", "PASS"),
    ("Total SRAM footprint",
     f"{total/1024:.2f} KB", "< 256 KB", "PASS"),
    ("ESP32 memory budget",
     f"{total/1024:.2f} / 520 KB", "< 520 KB", "PASS"),
    ("Raspberry Pi Pico memory budget",
     f"{total/1024:.2f} / 264 KB", "< 264 KB", "PASS"),
    ("Arduino Nano memory budget",
     f"{total/1024:.2f} / 256 KB", "< 256 KB",
     "PASS" if total/1024 < 256 else "FAIL"),
    ("Weighted F1 (PTQ model)",
     "0.8891", ">= 0.85", "PASS"),
    ("DDoS F1",
     "1.0000", ">= 0.95", "PASS"),
    ("Reconnaissance F1 (updated)",
     "0.66", ">= 0.50", "PASS"),
]
for name, val, target, result in rows:
    print(f"{name:<35} {val:<15} {target:<10} {result}")

PERFORMANCE BENCHMARKING SUMMARY — CB011911
Metric                              Value           Target     Result
-------------------------------------------------------
TFLite inference time (mean)        0.0099 ms       < 10 ms    PASS
TFLite inference time (max)         3.7646 ms       < 50 ms    PASS
XAI rule overhead (mean)            0.009177 ms     Negligible PASS
model.tflite size                   2.80 KB         < 50 KB    PASS
Total SRAM footprint                39.79 KB        < 256 KB   PASS
ESP32 memory budget                 39.79 / 520 KB  < 520 KB   PASS
Raspberry Pi Pico memory budget     39.79 / 264 KB  < 264 KB   PASS
Arduino Nano memory budget          39.79 / 256 KB  < 256 KB   PASS
Weighted F1 (PTQ model)             0.8891          >= 0.85    PASS
DDoS F1                             1.0000          >= 0.95    PASS
Reconnaissance F1 (updated)         0.66            >= 0.50    PASS
